In [ ]:
!pip install -q transformers accelerate bitsandbytes gradio requests



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
# import json
# import torch
import json
import re
import torch
import requests


ANALYTICS_URL = "https://a27a-2401-4900-30d3-fa7-b4b5-d682-cf4-608d.ngrok-free.app"

def extract_first_json(text):
    """
    Extract full JSON object by locating first '{' and last '}'.
    Handles nested JSON safely.
    """
    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in model output:\n{text}")

    json_str = text[start:end+1]

    try:
        return json.loads(json_str)
    except Exception as e:
        raise ValueError(f"Invalid JSON structure:\n{json_str}\nError: {e}")

def generate_query_object(question):

    prompt = f"""
You are a semantic query planner for a license analytics system.

Return ONLY valid JSON.
No explanation.
No markdown.
No extra text.

Schema:
{{
  "raw_question": "",
  "intent": "",
  "metric": "",
  "software": [],
  "feature": [],
  "time_range": {{
    "type": "",
    "value": ""
  }},
  "aggregation_scope": "",
  "group_by": "",
  "comparison": false,
  "filters": {{}}
}}

Rules:
- intent: current, trend, compare, anomaly, forecast, analysis
- metric: latest, average, sum, max, min, percent_change, volatility
- time_range.type: relative or absolute
- relative examples: "7d", "14d", "30d", "90d"
- absolute examples: "2026-02-14", "2026-02-14T09:30:00"

User question:
{question}
"""

    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("\nRAW MODEL OUTPUT:\n", generated_text)

    try:
        query_object = extract_first_json(generated_text)
        # query_object["raw_question"] = question
    except Exception as e:
        print("JSON extraction failed:", e)
        query_object = {}

    # HARD DEFAULTS
    if not isinstance(query_object, dict):
        query_object = {}

    query_object.setdefault("intent", "current")
    query_object.setdefault("metric", "latest")
    query_object.setdefault("software", [])
    query_object.setdefault("feature", [])
    query_object.setdefault("time_range", {"type": "relative", "value": "7d"})
    query_object.setdefault("aggregation_scope", "business_hours")
    query_object.setdefault("group_by", "none")
    query_object.setdefault("comparison", False)
    query_object.setdefault("filters", {})

    return query_object


In [ ]:
import requests
import torch
import json

ANALYTICS_URL = "https://a27a-2401-4900-30d3-fa7-b4b5-d682-cf4-608d.ngrok-free.app"

def analyze_question(question):

    # Step 1: Generate structured query
    try:
        query_object = generate_query_object(question)
        query_object["raw_question"] = question
        print("\nGenerated Query Object:")
        print(json.dumps(query_object, indent=2))
    except Exception as e:
        return f"SLM JSON generation error:\n{str(e)}"

    # Step 2: Call backend
    try:
        response = requests.post(
            f"{ANALYTICS_URL}/analyze",
            json=query_object,
            headers={"ngrok-skip-browser-warning": "true"},
            timeout=15
        )

        if response.status_code != 200:
            return f"Backend returned {response.status_code}:\n{response.text}"

        analytics_data = response.json()

    except Exception as e:
        return f"Backend communication error:\n{str(e)}"

    print("\nAnalytics Data:")
    print(json.dumps(analytics_data, indent=2))

    # Step 3: Deterministic explanation
    explanation_prompt = f"""
You are a deterministic license analytics interpreter.

STRICT RULES:
- Use ONLY provided analytics result.
- Do NOT assume missing data.
- Do NOT hallucinate.
- Do NOT repeat JSON.

User Question:
{question}

Analytics Result:
{analytics_data}

Explain clearly and professionally.
If numbers are present compute percentages.
"""

    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = tokenizer(
        explanation_prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=600,
            temperature=0.2,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    explanation = tokenizer.decode(outputs[0], skip_special_tokens=True)

    explanation = explanation.replace(explanation_prompt, "").strip()

    if explanation == "":
        explanation = "No explanation could be generated."

    return explanation


In [ ]:
import gradio as gr

def chat_interface(question):
    if not question or question.strip() == "":
        return "Please enter a valid question."

    try:
        return analyze_question(question)
    except Exception as e:
        return f"Unexpected error:\n{str(e)}"


demo = gr.Interface(
    fn=chat_interface,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Ask about license usage, trends, comparisons, or forecasts...",
        label="Your Question"
    ),
    outputs=gr.Textbox(
        lines=25,
        max_lines=40,
        show_copy_button=True,
        label="Analytics Response"
    ),
    title="📊 License Analytics Intelligence System",
    description="Ask detailed analytical questions about software license usage, trends, anomalies, and forecasts.",
    theme="soft"
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc14f36e496534154f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
